# Kinematics: Theta2 and Theta3

For any given Altitude and Roll pointing orientation, the inverse kinematics calculates the required Theta2 and Theta3 motor angles to achive the desired result. 

This notebook analyses the relationships between Altitude, Roll and Theta2, Theta3.


In [78]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import os, sys
import importlib
sys.path.insert(0, '../driver')
import pointing_model 
importlib.reload(pointing_model)
from pointing_model import azaltroll_to_theta, theta_to_azaltroll


Simulate a grid of pointing orientations spanning IK Altitude and Roll as well as FK Theta2 and Theta3 positions at a given Az

In [93]:

az = 180
d = []
for y in range(-80,81,10):
    for x in range(-80,81,10):
        f_t2 = x
        t1, t2, t3 = azaltroll_to_theta(az, x, y)
        _, alt, roll = theta_to_azaltroll(az, x, y)
        if x<-8:
            alt, roll, f_t2 = 0, 0, 0
        d.append({ 
            "i_alt": x, "i_roll": y, "i_theta2": t2, "i_theta3": t3,
            "f_alt": alt, "f_roll": roll, "f_theta2": f_t2, "f_theta3": y,
        })
d=pd.DataFrame(d)
d.columns


Index(['i_alt', 'i_roll', 'i_theta2', 'i_theta3', 'f_alt', 'f_roll',
       'f_theta2', 'f_theta3'],
      dtype='object')

# Altitude vs Theta2 and Theta3


In [88]:
fig = make_subplots(
    rows=1, 
    cols=2,
    subplot_titles=("Altitude vs Theta2", "Altitude vs Theta3")
)

# First plot (theta2)
fig1 = px.line(d, x="i_theta2", y="i_alt", color="i_roll")
for trace in fig1.data:
    trace.legendgroup = trace.name
    trace.showlegend = True   # Only show legend once
    fig.add_trace(trace, row=1, col=1)

# Second plot (theta3)
fig2 = px.line(d, x="i_theta3", y="i_alt", color="i_roll")
for trace in fig2.data:
    trace.legendgroup = trace.name
    trace.showlegend = False  # Hide duplicate legend entries
    fig.add_trace(trace, row=1, col=2)

# Adjust overall layout
fig.update_layout(height=800, width=1600, showlegend=True, legend_title_text="Roll (deg)")

fig.update_xaxes(title_text="Theta 3 (deg)", row=1, col=2)
fig.update_xaxes(title_text="Theta 2 (deg)", row=1, col=1)

fig.update_yaxes(title_text="Altitude (deg)", row=1, col=1)
fig.update_yaxes(title_text="Altitude (deg)", row=1, col=2)

fig.show()

# Roll vs Theta2 and Theta3

In [82]:
fig = make_subplots(
    rows=1, 
    cols=2,
    subplot_titles=("Roll vs Theta2", "Roll vs Theta3")
)

# First plot (theta2)
fig1 = px.line(d, x="i_theta2", y="i_roll", color="i_alt")
for trace in fig1.data:
    trace.legendgroup = trace.name
    trace.showlegend = True   # Only show legend once
    fig.add_trace(trace, row=1, col=1)

# Second plot (theta3)
fig2 = px.line(d, x="i_theta3", y="i_roll", color="i_alt")
for trace in fig2.data:
    trace.legendgroup = trace.name
    trace.showlegend = False  # Hide duplicate legend entries
    fig.add_trace(trace, row=1, col=2)

# Adjust overall layout
fig.update_layout(height=800, width=1600, showlegend=True, legend_title_text="Altitude (deg)")

fig.update_xaxes(title_text="Theta 3 (deg)", row=1, col=2)
fig.update_xaxes(title_text="Theta 2 (deg)", row=1, col=1)

fig.update_yaxes(title_text="Roll (deg)", row=1, col=1)
fig.update_yaxes(title_text="Roll (deg)", row=1, col=2)

fig.show()

# Altitude and Roll isobars in the Theta2/Theta3 space

In [84]:
fig = go.Figure()

# --- SOLID LINES: grouped by alt ---
for val in d["i_alt"].unique():
    df_sub = d[d["i_alt"] == val]
    fig.add_trace(
        go.Scatter(x=df_sub["i_theta3"], y=df_sub["i_theta2"], name=f"alt={val}",
            mode="lines", line=dict(dash="solid"), legendgroup="alt", showlegend=True)
    )

# --- DASHED LINES: grouped by roll ---
for val in d["i_roll"].unique():
    df_sub = d[d["i_roll"] == val]
    fig.add_trace(
        go.Scatter(x=df_sub["i_theta3"], y=df_sub["i_theta2"], name=f"roll={val}",
            mode="lines", line=dict(dash="dash"), legendgroup="roll", showlegend=True)
    )

fig.update_layout(
    height=900, width=1600, xaxis_title="Theta3 (deg)", yaxis_title="Theta2 (deg)", legend_title_text="Grouping"
)

fig.show()

# Theta2 and Theta3 isobars in the Altitude/Roll  space

In [98]:
fig = go.Figure()

# --- SOLID LINES: grouped by alt ---
for val in d["f_theta2"].unique():
    df_sub = d[d["f_theta2"] == val]
    fig.add_trace(
        go.Scatter(x=df_sub["f_roll"], y=df_sub["f_alt"], name=f"theta2={val}",
            mode="lines", line=dict(dash="solid"), legendgroup="alt", showlegend=True)
    )

# --- DASHED LINES: grouped by roll ---
for val in d["f_theta3"].unique():
    df_sub = d[d["f_theta3"] == val]
    fig.add_trace(
        go.Scatter(x=df_sub["f_roll"], y=df_sub["f_alt"], name=f"theta3={val}",
            mode="lines", line=dict(dash="dash"), legendgroup="roll", showlegend=True)
    )

fig.update_layout(
    height=900, width=1400, xaxis_title="Roll (deg)", yaxis_title="Altitude (deg)", legend_title_text="Grouping"
)

fig.show()

# Scatter Matrix of Altitude, Roll, Theta2 and Theta3

In [42]:
fig = px.scatter_matrix(d, dimensions=["alt", "roll", "theta2", "theta3"], color="roll")
fig.update_layout(height=1000,width=1000 )
fig.show()